In [7]:
import gc
import time
import numpy as np
import pandas as pd
from datetime import datetime

In [8]:
# Data loading
train = pd.read_csv('./elo-merchant-category-recommendation/train_pre.csv', header=0)
test = pd.read_csv('./elo-merchant-category-recommendation/test_pre.csv', header=0)
transaction = pd.read_csv('./elo-merchant-category-recommendation/transaction_d_pre.csv', header=0)

In [9]:
numeric_cols = ['purchase_amount', 'installments']

category_cols = ['authorized_flag', 'city_id', 'category_1',
       'category_3', 'merchant_category_id','month_lag','most_recent_sales_range',
                 'most_recent_purchases_range', 'category_4',
                 'purchase_month', 'purchase_hour_section', 'purchase_day']

id_cols = ['card_id', 'merchant_id']

In [10]:
# Feature Construction
features = {}
card_all = pd.concat([train['card_id'],test['card_id']]).values.tolist()
for card in card_all:
    features[card] = {}
     
columns = transaction.columns.tolist()
idx = columns.index('card_id')
category_cols_index = [columns.index(col) for col in category_cols]
numeric_cols_index = [columns.index(col) for col in numeric_cols]

s = time.time()
num = 0

for i in range(transaction.shape[0]):
    va = transaction.loc[i].values
    card = va[idx]
    for cate_ind in category_cols_index:
        for num_ind in numeric_cols_index:
            col_name = '&'.join([str(columns[cate_ind]), str(va[cate_ind]), str(columns[num_ind])])
            features[card][col_name] = features[card].get(col_name, 0) + va[num_ind]
    num += 1
    if num%1000000==0:
        print(time.time()-s, "s")
del transaction
gc.collect()

25.9537410736084 s
52.28138017654419 s
78.17697310447693 s
104.24861693382263 s
130.8835163116455 s
156.24182415008545 s
181.24815702438354 s
206.52079701423645 s
232.7751979827881 s
258.2357840538025 s
283.99987721443176 s
309.1496660709381 s
334.5746839046478 s
359.9380979537964 s
385.4452760219574 s
412.2400541305542 s
437.7047691345215 s
463.42817902565 s
489.17707324028015 s
515.6401662826538 s
542.0333161354065 s
568.6558530330658 s
595.0728199481964 s
620.9728813171387 s
647.3791749477386 s
673.5057952404022 s
699.4539849758148 s
725.566015958786 s
752.1636800765991 s
779.8282351493835 s
808.5201680660248 s


901

In [ ]:
# Covert features dictionary to a dataframe
df = pd.DataFrame(features).T.reset_index()
cols = df.columns.tolist()
df.columns = ['card_id'] + cols[1:]

# Merge features to train and test datasets separately
train = pd.merge(train, df, how='left', on='card_id')
test =  pd.merge(test, df, how='left', on='card_id')

train.to_csv("./elo-merchant-category-recommendation/train_dict.csv", index=False)
test.to_csv("./elo-merchant-category-recommendation/test_dict.csv", index=False)

gc.collect()

0

In [12]:
del features
del df

In [14]:
transaction = pd.read_csv('./elo-merchant-category-recommendation/transaction_g_pre.csv')

In [15]:
numeric_cols = ['authorized_flag',  'category_1', 'installments',
       'category_3',  'month_lag','purchase_month','purchase_day','purchase_day_diff', 'purchase_month_diff',
       'purchase_amount', 'category_2', 
       'purchase_month', 'purchase_hour_section', 'purchase_day',
       'most_recent_sales_range', 'most_recent_purchases_range', 'category_4']
categorical_cols = ['city_id', 'merchant_category_id', 'merchant_id', 'state_id', 'subsector_id']

In [16]:
aggs = {}

for col in numeric_cols:
    aggs[col] = ['nunique', 'mean', 'min', 'max', 'var', 'skew', 'sum']
for col in categorical_cols:
    aggs[col] = ['nunique']
aggs['card_id'] = ['size', 'count']

cols = ['card_id']

for key in aggs.keys():
    cols.extend([key+''+stat for stat in aggs[key]])

# Merge historical and new transactions
df = transaction[transaction['month_lag'] < 0].groupby('card_id').agg(aggs).reset_index()
df.columns = cols[:1] + [co+'hist' for co in cols[1:]]

df2 = transaction[transaction['month_lag']>=0].groupby('card_id').agg(aggs).reset_index()
df2.columns = cols[:1] + [co+'_new' for co in cols[1:]]
df = pd.merge(df, df2, how='left', on='card_id')

# Merge overall transaction
df2= transaction.groupby('card_id').agg(aggs).reset_index()
df2.columns = cols
df = pd.merge(df, df2, how='left', on='card_id')
del transaction
gc.collect()


train =  pd.merge(train, df, how='left', on='card_id')
test  = pd.merge(test, df, how='left', on='card_id')
del df
train.to_csv("./elo-merchant-category-recommendation/train_groupby.csv", index=False)
test.to_csv("./elo-merchant-category-recommendation/test_groupby.csv", index=False)

gc.collect()

0

In [25]:
# Load all preprocessed datasets
train_dict = pd.read_csv("./elo-merchant-category-recommendation/train_dict.csv")
test_dict = pd.read_csv("./elo-merchant-category-recommendation/test_dict.csv")
train_groupby = pd.read_csv("./elo-merchant-category-recommendation/train_groupby.csv")
test_groupby = pd.read_csv("./elo-merchant-category-recommendation/test_groupby.csv")

In [33]:
# Drop duplicate columns
for col in train_dict.columns:
    if col in train_groupby.columns and col != 'card_id':
        del train_groupby[col]

for col in test_dict.columns:
    if col in test_groupby.columns and col != 'card_id':
        del test_groupby[col]

In [34]:
train = pd.merge(train_dict, train_groupby, how='left', on='card_id').fillna(0)
test = pd.merge(test_dict, test_groupby, how='left', on='card_id').fillna(0)

In [35]:
train.to_csv("./elo-merchant-category-recommendation/train.csv", index=False)
test.to_csv("./elo-merchant-category-recommendation/test.csv", index=False)

del train_dict, test_dict, train_groupby, test_groupby
gc.collect()

0